In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

org = pd.read_excel(
    "../data/Organization.xlsx"
)

print("Shape:", org.shape)

location_keywords = [
    "lat",
    "lon",
    "address",
    "city",
    "zip",
    "county",
    "geoid",
    "block",
    "tract"
]

location_columns = []

for column in org.columns:
    column_lower = column.lower()

    for keyword in location_keywords:
        if keyword in column_lower:
            location_columns.append(column)
            break

print("\nPossible location columns:")
print(location_columns)

display(
    org[
        location_columns
    ].head(10)
)

print("\nMissing values:")
print(
    org[
        location_columns
    ].isna().sum()
)

Shape: (1202, 68)

Possible location columns:
['City', 'Zip', 'MailCity', 'MailZip', 'County', 'block_geoid', 'is_zip_centroid']


,City,Zip,MailCity,MailZip,County,block_geoid,is_zip_centroid
0,Rochester,55904.0,Rochester,55904,Olmsted,2.710900e+14,0.0
1,Brooklyn Park,55428.0,NaN,NaN,Hennepin-Suburbs,2.705303e+14,0.0
2,East Grand Forks,56721.0,East Grand Forks,56721,Polk,2.711902e+14,0.0
3,Duluth,55808.0,Duluth,55808,Saint Louis,2.713700e+14,0.0
4,Fargo,58102.0,Fargo,58102,Clay,3.801700e+14,0.0
5,Anoka,55303.0,Anoka,55303,Hennepin-Suburbs,2.700305e+14,0.0
6,Albany,56307.0,Albany,56307,Stearns,2.714501e+14,0.0
7,Minneapolis,55409.0,Minneapolis,55409,Hennepin-Minneapolis,2.705311e+14,0.0
8,Rice,56367.0,Rice,56367,Benton,2.700902e+14,0.0
9,Annandale,55302.0,Annandale,55302,Wright,2.717110e+14,0.0



Missing values:
City               141
Zip                148
MailCity           293
MailZip            293
County              62
block_geoid        150
is_zip_centroid    150
dtype: int64


In [2]:
activity = pd.read_csv(
    "../data/processed/site_month_clean.csv",
    low_memory=False
)

print("Activity shape:", activity.shape)
print("Unique site IDs:", activity["_id_site"].nunique())

Activity shape: (28154, 65)
Unique site IDs: 633


In [3]:
activity_2024 = activity[
    (activity["_Year"] == 2024)
    & (activity["County"].notna())
    & (activity["stable_type"] != "Meal Program")
].copy()

print("2024 rows:", len(activity_2024))
print("2024 active sites:", activity_2024["_id_site"].nunique())

2024 rows: 5619
2024 active sites: 484


In [4]:
site_supply = (
    activity_2024
    .groupby("_id_site", as_index=False)
    .agg(
        annual_visits=("HouseholdsReg", "sum"),
        annual_individuals=("IndividualsReg", "sum"),
        annual_pounds=("PoundsReg", "sum"),
        activity_county=("County", "first"),
        stable_type=("stable_type", "first"),
        food_bank=("FoodBank", "first")
    )
)

print("Supply sites:", len(site_supply))

display(
    site_supply.head()
)

Supply sites: 484


,_id_site,annual_visits,annual_individuals,annual_pounds,activity_county,stable_type,food_bank
0,1066.0,26353.0,74688.0,1317093.0,Hennepin,Food Shelf,Second Harvest Heartland
1,1067.0,438.0,945.0,47171.0,Stearns,Food Shelf,Second Harvest Heartland
2,1069.0,4008.0,5768.0,282968.0,Hennepin,Food Shelf,The Food Group
3,1070.0,527.0,1233.0,44795.0,Benton,Food Shelf,Second Harvest Heartland
4,1071.0,3604.0,8445.0,376090.0,Wright,Food Shelf,Second Harvest Heartland


In [5]:
site_supply["_id_site"] = pd.to_numeric(site_supply["_id_site"], errors="coerce").fillna(0).astype("int64")
org["__ID"] = pd.to_numeric(org["__ID"], errors="coerce").fillna(0).astype("int64")


In [6]:
print(
    "Organization IDs unique:",
    org["__ID"].is_unique
)

print(
    "Duplicate organization IDs:",
    org["__ID"].duplicated().sum()
)

Organization IDs unique: False
Duplicate organization IDs: 5


In [8]:
print(
    "Supply IDs unique:",
    site_supply["_id_site"].is_unique
)

print(
    "Missing supply IDs:",
    site_supply["_id_site"].isna().sum()
)

print(
    "\nMissing numeric organization IDs:",
    org["__ID"].isna().sum()
)

numeric_org = org[
    org["__ID"].notna()
].copy()

duplicate_org_ids = numeric_org[
    numeric_org["__ID"].duplicated(
        keep=False
    )
].copy()

print(
    "Duplicate NON-MISSING organization IDs:",
    duplicate_org_ids["__ID"].nunique()
)

print(
    "Rows belonging to duplicate IDs:",
    len(duplicate_org_ids)
)

display(
    duplicate_org_ids[
        [
            "__ID",
            "City",
            "Zip",
            "County",
            "block_geoid",
            "is_zip_centroid"
        ]
    ]
    .sort_values("__ID")
    .head(30)
)

Supply IDs unique: True
Missing supply IDs: 0

Missing numeric organization IDs: 0
Duplicate NON-MISSING organization IDs: 5
Rows belonging to duplicate IDs: 10


,__ID,City,Zip,County,block_geoid,is_zip_centroid
557,14694,Minneapolis,55405.0,Hennepin-Minneapolis,2.705313e+14,0.0
589,14694,St. Paul,55118.0,Dakota,2.703706e+14,0.0
566,14696,Burnsville,55337.0,Dakota,2.703706e+14,0.0
590,14696,Minneapolis,55418.0,Hennepin-Minneapolis,2.705310e+14,0.0
569,14699,NaN,NaN,NaN,NaN,NaN
591,14699,Austin,55912.0,Mower,2.709900e+14,0.0
570,14703,NaN,NaN,Lake,NaN,NaN
592,14703,Brooklyn Center,55430.0,Hennepin-Suburbs,2.705302e+14,0.0
574,14710,NaN,NaN,St. Louis,NaN,NaN
595,14710,Minneapolis,55407.0,Hennepin-Minneapolis,2.705311e+14,0.0


In [9]:
duplicate_ids = duplicate_org_ids["__ID"].unique()

display(
    site_supply[
        site_supply["_id_site"].isin(duplicate_ids)
    ][
        [
            "_id_site",
            "activity_county",
            "annual_visits",
            "stable_type",
            "food_bank"
        ]
    ]
    .sort_values("_id_site")
)

,_id_site,activity_county,annual_visits,stable_type,food_bank
385,14694,Dakota,2773.0,Food Shelf / Outreach,Second Harvest Heartland
386,14696,Hennepin,883.0,Food Shelf,The Food Group
387,14699,Mower,4670.0,Mobile / Outreach,Channel One Regional Food Bank*
388,14703,Hennepin,0.0,Food Shelf,The Food Group
392,14710,Hennepin,19620.0,Food Shelf,The Food Group


In [10]:
org_check = duplicate_org_ids.copy()

org_check["org_county_clean"] = (
    org_check["County"]
    .replace({
        "Hennepin-Minneapolis": "Hennepin",
        "Hennepin-Suburbs": "Hennepin",
        "Ramsey-Saint Paul": "Ramsey",
        "Ramsey-Suburbs": "Ramsey",
        "Saint Louis": "St. Louis",
        "Lac Qui Parle": "Lac qui Parle"
    })
)

In [11]:
duplicate_check = site_supply[
    site_supply["_id_site"].isin(duplicate_ids)
][
    [
        "_id_site",
        "activity_county",
        "annual_visits"
    ]
].merge(
    org_check[
        [
            "__ID",
            "City",
            "Zip",
            "County",
            "org_county_clean",
            "block_geoid",
            "is_zip_centroid"
        ]
    ],
    left_on="_id_site",
    right_on="__ID",
    how="left"
)

duplicate_check["county_match"] = (
    duplicate_check["activity_county"]
    == duplicate_check["org_county_clean"]
)

display(
    duplicate_check[
        [
            "_id_site",
            "activity_county",
            "City",
            "Zip",
            "County",
            "org_county_clean",
            "block_geoid",
            "county_match"
        ]
    ]
    .sort_values(["_id_site", "county_match"])
)

,_id_site,activity_county,City,Zip,County,org_county_clean,block_geoid,county_match
0,14694,Dakota,Minneapolis,55405.0,Hennepin-Minneapolis,Hennepin,2.705313e+14,False
1,14694,Dakota,St. Paul,55118.0,Dakota,Dakota,2.703706e+14,True
2,14696,Hennepin,Burnsville,55337.0,Dakota,Dakota,2.703706e+14,False
3,14696,Hennepin,Minneapolis,55418.0,Hennepin-Minneapolis,Hennepin,2.705310e+14,True
4,14699,Mower,NaN,NaN,NaN,NaN,NaN,False
5,14699,Mower,Austin,55912.0,Mower,Mower,2.709900e+14,True
6,14703,Hennepin,NaN,NaN,Lake,Lake,NaN,False
7,14703,Hennepin,Brooklyn Center,55430.0,Hennepin-Suburbs,Hennepin,2.705302e+14,True
8,14710,Hennepin,NaN,NaN,St. Louis,St. Louis,NaN,False
9,14710,Hennepin,Minneapolis,55407.0,Hennepin-Minneapolis,Hennepin,2.705311e+14,True


In [12]:
org_resolved = org.copy()

org_resolved["org_county_clean"] = (
    org_resolved["County"]
    .replace({
        "Hennepin-Minneapolis": "Hennepin",
        "Hennepin-Suburbs": "Hennepin",
        "Ramsey-Saint Paul": "Ramsey",
        "Ramsey-Suburbs": "Ramsey",
        "Saint Louis": "St. Louis",
        "Lac Qui Parle": "Lac qui Parle"
    })
)

In [13]:
duplicate_id_mask = org_resolved[
    "__ID"
].duplicated(keep=False)

org_unique = org_resolved[
    ~duplicate_id_mask
].copy()

org_duplicates = org_resolved[
    duplicate_id_mask
].copy()

print(
    "Unique-ID rows:",
    len(org_unique)
)

print(
    "Duplicate-ID rows:",
    len(org_duplicates)
)

Unique-ID rows: 1192
Duplicate-ID rows: 10


In [14]:
duplicate_activity_counties = site_supply[
    site_supply["_id_site"].isin(
        duplicate_ids
    )
][
    [
        "_id_site",
        "activity_county"
    ]
].copy()

org_duplicates = org_duplicates.merge(
    duplicate_activity_counties,
    left_on="__ID",
    right_on="_id_site",
    how="left"
)

In [15]:
resolved_duplicates = org_duplicates[
    org_duplicates["org_county_clean"]
    == org_duplicates["activity_county"]
].copy()

print(
    "Resolved duplicate IDs:",
    resolved_duplicates["__ID"].nunique()
)

display(
    resolved_duplicates[
        [
            "__ID",
            "activity_county",
            "City",
            "Zip",
            "County",
            "block_geoid"
        ]
    ]
)

Resolved duplicate IDs: 5


,__ID,activity_county,City,Zip,County,block_geoid
5,14694,Dakota,St. Paul,55118.0,Dakota,2.703706e+14
6,14696,Hennepin,Minneapolis,55418.0,Hennepin-Minneapolis,2.705310e+14
7,14699,Mower,Austin,55912.0,Mower,2.709900e+14
8,14703,Hennepin,Brooklyn Center,55430.0,Hennepin-Suburbs,2.705302e+14
9,14710,Hennepin,Minneapolis,55407.0,Hennepin-Minneapolis,2.705311e+14


In [16]:
resolved_duplicates = resolved_duplicates.drop(
    columns=[
        "_id_site",
        "activity_county"
    ]
)

org_for_merge = pd.concat(
    [
        org_unique,
        resolved_duplicates
    ],
    ignore_index=True
)

print(
    "Organization IDs unique:",
    org_for_merge["__ID"].is_unique
)

print(
    "Rows:",
    len(org_for_merge)
)

Organization IDs unique: True
Rows: 1197


In [17]:
site_locations = site_supply.merge(
    org_for_merge[
        [
            "__ID",
            "City",
            "Zip",
            "County",
            "org_county_clean",
            "block_geoid",
            "is_zip_centroid"
        ]
    ],
    left_on="_id_site",
    right_on="__ID",
    how="left",
    validate="one_to_one"
)

print(
    "Supply sites:",
    len(site_supply)
)

print(
    "Merged rows:",
    len(site_locations)
)

print(
    "\nMissing organization match:",
    site_locations["__ID"].isna().sum()
)

print(
    "Missing block GEOID:",
    site_locations["block_geoid"].isna().sum()
)

print(
    "Missing ZIP:",
    site_locations["Zip"].isna().sum()
)

Supply sites: 484
Merged rows: 484

Missing organization match: 0
Missing block GEOID: 11
Missing ZIP: 29


In [18]:
site_locations["block_geoid_text"] = (
    pd.to_numeric(
        site_locations["block_geoid"],
        errors="coerce"
    )
    .round()
    .astype("Int64")
    .astype("string")
)

site_locations["state_fips"] = (
    site_locations["block_geoid_text"]
    .str[:2]
)

print(
    site_locations[
        "state_fips"
    ].value_counts(
        dropna=False
    )
)

state_fips
27      473
<NA>     11
Name: count, dtype: Int64


In [19]:
missing_geo = site_locations[
    site_locations["block_geoid"].isna()
].copy()

display(
    missing_geo[
        [
            "_id_site",
            "activity_county",
            "City",
            "Zip",
            "County",
            "annual_visits",
            "annual_individuals",
            "annual_pounds",
            "stable_type",
            "food_bank",
            "is_zip_centroid"
        ]
    ]
    .sort_values(
        "annual_visits",
        ascending=False
    )
)

,_id_site,activity_county,City,Zip,County,annual_visits,annual_individuals,annual_pounds,stable_type,food_bank,is_zip_centroid
324,14468,Ramsey,NaN,NaN,Ramsey-Suburbs,1362.0,3060.0,53667.0,Food Shelf / Outreach,Second Harvest Heartland,NaN
164,1284,St. Louis,NaN,NaN,St. Louis,640.0,1323.0,31972.0,Food Shelf,Second Harvest Northland,NaN
151,1266,Lake,NaN,NaN,Lake,473.0,1095.0,28373.0,Food Shelf,Second Harvest Northland,NaN
406,14763,Koochiching,International Falls,56649.0,Koochiching,421.0,379.0,1491.0,Food Shelf,Second Harvest Northland,NaN
423,14817,Hennepin,NaN,NaN,Hennepin-Suburbs,219.0,219.0,12045.0,Food Shelf,Second Harvest Heartland,NaN
285,14169,Hennepin,NaN,NaN,Hennepin-Suburbs,0.0,0.0,0.0,Food Shelf,Second Harvest Heartland,NaN
421,14811,Hennepin,Minneapolis,55423.0,Hennepin-Minneapolis,0.0,0.0,0.0,Food Shelf,The Food Group,NaN
425,14912,Aitkin,Palisade,56431.0,Aitkin,0.0,0.0,0.0,Mobile / Outreach,Second Harvest Northland,NaN
450,15324,Hennepin,NaN,NaN,Hennepin-Suburbs,0.0,0.0,0.0,Food Shelf,Second Harvest Heartland,NaN
457,15342,Clay,NaN,NaN,Clay,0.0,0.0,0.0,Food Shelf,Great Plains Food Bank,NaN


In [20]:
total_visits = site_locations["annual_visits"].sum()

missing_geo_visits = missing_geo[
    "annual_visits"
].sum()

missing_geo_share = (
    missing_geo_visits
    / total_visits
) * 100

print(
    "Sites missing block GEOID:",
    len(missing_geo)
)

print(
    "Annual visits from missing-location sites:",
    missing_geo_visits
)

print(
    "Share of all annual visits:",
    missing_geo_share
)

print(
    "\nMissing-location sites with ZIP:",
    missing_geo["Zip"].notna().sum()
)

print(
    "Missing-location sites with City:",
    missing_geo["City"].notna().sum()
)

Sites missing block GEOID: 11
Annual visits from missing-location sites: 3115.0
Share of all annual visits: 0.10958065611197981

Missing-location sites with ZIP: 3
Missing-location sites with City: 3


In [21]:
print(
    "Sites with zero annual visits:",
    (site_locations["annual_visits"] == 0).sum()
)

print(
    "Missing-location sites with zero annual visits:",
    (missing_geo["annual_visits"] == 0).sum()
)

Sites with zero annual visits: 12
Missing-location sites with zero annual visits: 6
